# 03 — Physical feature extraction (the novelty's input)

Partner B, week 2. Before running this on real data, make sure
`pytest tests/test_features_synthetic.py` passes locally (pure NumPy, no GPU needed) —
that's the ground-truth check on CFO/I-Q-imbalance/phase-noise recovery.

In [ ]:
!python -m pytest tests/test_features_synthetic.py -q

In [ ]:
!python -m src.features.extract_all --config configs/default.yaml \
    --split data/splits/split_seed0.json --out results/features/features.csv

## Sanity check: do features cluster by device, or by session/receiver?

If they cluster by session/receiver instead of device, the consistency gate would be
learning environment, not hardware — flag this now (see PAPER_NOTES.md limitations).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.features.pipeline import FEATURE_COLUMNS
from src.features.session_normalize import add_session_relative_features

all_features = pd.read_csv('results/features/features.csv')
all_features = add_session_relative_features(all_features, FEATURE_COLUMNS)

# Left: raw CFO vs amplitude imbalance coloured by device -- expect receiver-driven smearing.
# Right: the SAME two features after session-relative correction -- device clusters should
# tighten. If they still cluster by session_id instead of tx_id, the gate is learning
# environment, not hardware (report that as a limitation, see PAPER_NOTES.md).
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.scatterplot(data=all_features, x='cfo_hz', y='iq_amp_imbalance_db', hue='tx_id', legend=False, ax=axes[0], s=6)
axes[0].set_title('RAW features, coloured by device')
sns.scatterplot(data=all_features, x='cfo_hz_rel', y='iq_amp_imbalance_db_rel', hue='tx_id', legend=False, ax=axes[1], s=6)
axes[1].set_title('SESSION-RELATIVE features, coloured by device')
sns.scatterplot(data=all_features, x='cfo_hz_rel', y='iq_amp_imbalance_db_rel', hue='session_id', legend=False, ax=axes[2], s=6)
axes[2].set_title('SESSION-RELATIVE features, coloured by session (day+rx)')
plt.savefig('results/features/device_vs_session_clustering.png', dpi=150, bbox_inches='tight')
plt.show()